In [71]:
import pandas as pd
data = pd.read_csv('data/students.csv')
data = data.dropna() # opcionalno
data.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72.0,72.0,74.0
2,female,group B,master's degree,standard,none,90.0,95.0,93.0
4,male,group C,some college,standard,none,76.0,78.0,75.0
6,female,group B,some college,standard,completed,88.0,95.0,92.0
8,male,group D,high school,free/reduced,completed,64.0,64.0,67.0


In [72]:
data.shape

(996, 8)

In [73]:
X_cols = ['gender', 'race/ethnicity', 'math score', 'reading score']
y_col = 'writing score'

In [74]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(data[X_cols], data[y_col], train_size=0.6)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, train_size=0.5)

In [75]:
from sklearn.preprocessing import LabelEncoder
le_gender = LabelEncoder()
le_race = LabelEncoder()

X_train['gender'] = le_gender.fit_transform(X_train['gender'])
X_train['race/ethnicity'] = le_race.fit_transform(X_train['race/ethnicity'])

In [76]:
X_test['gender'] = le_gender.transform(X_test['gender'])
X_val['gender'] = le_gender.transform(X_val['gender'])

X_test['race/ethnicity'] = le_race.transform(X_test['race/ethnicity'])
X_val['race/ethnicity'] = le_race.transform(X_val['race/ethnicity'])

In [77]:
X_train.head()

,gender,race/ethnicity,math score,reading score
74,1,2,49.0,49.0
192,0,1,62.0,64.0
115,1,2,84.0,77.0
219,1,1,61.0,56.0
424,1,1,41.0,39.0


In [78]:
import joblib
joblib.dump(le_gender, 'artifacts/le_gender.pkl')
joblib.dump(le_race, 'artifacts/le_race.pkl')

['artifacts/le_race.pkl']

In [79]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [80]:
joblib.dump(scaler, 'artifacts/std_scaler_X.pkl')

['artifacts/std_scaler_X.pkl']

In [81]:
y_train_2d = y_train.to_numpy().reshape(-1,1)
y_val_2d = y_val.to_numpy().reshape(-1,1)
y_test_2d = y_test.to_numpy().reshape(-1,1)

In [99]:
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train_2d)
y_val = scaler_y.transform(y_val_2d)
y_test = scaler_y.transform(y_test_2d)

In [100]:
joblib.dump(scaler_y, 'artifacts/std_scaler_y.pkl')

['artifacts/std_scaler_y.pkl']

In [85]:
# Prelazimo na drugi dio - izgradnja modela

In [88]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers

In [89]:
input_shape = X_train.shape[1]

In [90]:
basic_model = Sequential([
    layers.Input(shape=(input_shape,)),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='tanh'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

In [91]:
from keras.optimizers import Adam

basic_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mse", metrics=["mae"])

In [92]:
basic_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=32)

Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.6252 - mae: 0.6163 - val_loss: 0.1603 - val_mae: 0.3209
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1585 - mae: 0.3169 - val_loss: 0.1243 - val_mae: 0.2876
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1245 - mae: 0.2772 - val_loss: 0.0933 - val_mae: 0.2461
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1061 - mae: 0.2603 - val_loss: 0.0917 - val_mae: 0.2465
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1086 - mae: 0.2614 - val_loss: 0.0933 - val_mae: 0.2438
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1020 - mae: 0.2586 - val_loss: 0.0904 - val_mae: 0.2419
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1110 - mae: 0.2671 - val_loss: 0.0903 - val_mae: 0.2440
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1141 - mae: 0.2679 - val_loss: 0.0920 - val_mae: 0.2451
Epoch 9/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1087 - mae:

In [93]:
basic_model.evaluate(X_test, y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0816 - mae: 0.2299


[0.0815548226237297, 0.2298654019832611]

In [101]:
y_pred = basic_model.predict(X_test)
y_pred[:5]

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


array([[ 0.32648298],
       [ 0.575659  ],
       [ 0.16628182],
       [-0.00936596],
       [ 0.3811488 ]], dtype=float32)

In [102]:
y_pred = scaler_y.inverse_transform(y_pred)
y_pred[:5]

array([[72.71676 ],
       [76.434265],
       [70.32668 ],
       [67.70616 ],
       [73.53233 ]], dtype=float32)

In [103]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

advanced_model = Sequential([
    layers.Input(shape=(input_shape,)),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='tanh'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

In [105]:
advanced_model.compile(optimizer=Adam(0.01), loss="mse", metrics=["mae"])

In [107]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

In [108]:
advanced_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=1000, batch_size=32,
                   callbacks=[early_stopping])

Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 2.3410 - mae: 1.0030 - val_loss: 6.3361 - val_mae: 2.3827
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6909 - mae: 0.5902 - val_loss: 3.9599 - val_mae: 1.8793
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.4409 - mae: 0.4230 - val_loss: 1.5910 - val_mae: 1.1056
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3716 - mae: 0.3816 - val_loss: 0.9787 - val_mae: 0.8174
Epoch 5/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.3618 - mae: 0.3865 - val_loss: 0.9255 - val_mae: 0.8156
Epoch 6/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3060 - mae: 0.3522 - val_loss: 0.4787 - val_mae: 0.5246
Epoch 7/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2842 - mae: 0.3498 - val_loss: 0.2856 - val_mae: 0.3569
Epoch 8/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3022 - mae: 0.3634 - val_loss: 0.2514 - val_mae: 0.3257
Epoch 9/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.26

In [109]:
advanced_model.evaluate(X_test, y_test)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2003 - mae: 0.3270 


[0.20034393668174744, 0.3270398676395416]

In [110]:
y_pred = advanced_model.predict(X_test)
y_pred[:5]

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 


array([[ 0.22631872],
       [ 0.46245086],
       [ 0.08317474],
       [-0.21284477],
       [ 0.39730772]], dtype=float32)

In [111]:
y_pred = scaler_y.inverse_transform(y_pred)
y_pred[:5]

array([[71.22239],
       [74.74529],
       [69.08679],
       [64.67042],
       [73.77341]], dtype=float32)

In [112]:
advanced_model.save('artifacts/advanced_model.keras')